In [4]:
%env TORCHDYNAMO_DISABLE=1

env: TORCHDYNAMO_DISABLE=1


In [5]:
import numpy as np
from ase import units
from ase.md.langevin import Langevin
from aimnet.calculators import AIMNet2ASE
from arcskin.nanoreactor_run import SMILESPack,smiles_to_atoms
from arcskin.nanoreactor_analysis import AnalyseNanoReactor
import itertools as it 

INFO:rdkit:Enabling RDKit 2026.03.6 jupyter extensions


In [6]:
compounds = {
    "NO2":  "[N+](=O)[O-]",
    "NO":   "[N]=O",
    "O2":   "O=O",
}

reactants = ["[N]=O","[N]=O"]
reactants = ["[HH]","O=O"]
sp = SMILESPack(reactants)
cells, charges = sp.run_multiple(1,counts=[5,5],min_dist=2) # two of each molecule, with a minimum distance of 2\AA
cells

[15:12:06] WARNING: not removing hydrogen atom without neighbors


[Atoms(symbols='H10O10', pbc=False)]

In [7]:
class SphereWall:
    """V = sum_i m_i*k/2 * (r_i - r0)^2 for atoms beyond r0 from the origin."""
    def __init__(self, r0, k):
        self.r0, self.k = r0, k
    def _x(self, atoms):
        r = np.linalg.norm(atoms.positions, axis=1)
        return r, np.clip(r - self.r0, 0, None) * atoms.get_masses() * self.k
    def adjust_forces(self, atoms, forces):
        r, g = self._x(atoms)
        forces -= (g / np.maximum(r, 1e-9))[:, None] * atoms.positions
    def adjust_potential_energy(self, atoms):
        r, g = self._x(atoms)
        return 0.5 * np.sum(g * np.clip(r - self.r0, 0, None))
    def adjust_positions(self, atoms, new): pass
    def adjust_momenta(self, atoms, momenta): pass
    def get_removed_dof(self, atoms): return 0




In [8]:
#setup run 
def attach_calc_and_wall(atoms,filename="test.traj"):

    def squeeze(period=2000.0, t_open=5000.0):  # fs
        relaxed = (dyn.get_time() / units.fs) % period < t_open
        wall.r0, wall.k = (14.0, 1.0 * kc) if relaxed else (8.0, 0.5 * kc)

    kc = units.kcal / units.mol
    wall = SphereWall(14.0, 1.0 * kc)          # Å, kcal/mol/Å² per amu
    atoms.set_constraint(wall)
    atoms.calc = AIMNet2ASE("aimnet2")

    dyn = Langevin(atoms, 0.5 * units.fs, temperature_K=5000,
               friction=0.01 / units.fs, fixcm=False,trajectory=filename)


    dyn.attach(squeeze, interval=0.1)

    return dyn



atoms = cells[0]
dyn = attach_calc_and_wall(atoms)
# run nanoreactor 

dyn.run(100)

True

In [9]:
#analyse 
anr = AnalyseNanoReactor(skip_step=10,reactants=tuple(reactants))
anr.read_trajectory("test.traj",format=None)
anr.analyse_trajectory()

[15:12:11] WARNING: not removing hydrogen atom without neighbors
100%|██████████| 11/11 [00:03<00:00,  2.80it/s]


In [10]:
import pandas as pd 
pd.Series(anr.unique_smiles_counter[0])

[H][H]    55
dtype: int64

## see if we can predict H2O + NO2 + SO2 = H2SO4 + NO

In [12]:
compounds = ["[O]N=O","O=S=O","O"] # NO2, SO2, and H2O
combinations = list(it.combinations(compounds,2))

In [13]:
data = {}

for i,combination in enumerate(combinations): 
    # generate cells 
    sp = SMILESPack(combination)
    cells, charges = sp.run_multiple(1,counts=[5,5],min_dist=2) # two of each molecule, with a minimum distance of 2\AA
    atoms = cells[0]
    dyn = attach_calc_and_wall(atoms,filename=f"test_{i}.traj")

    dyn.run(5000)

    anr = AnalyseNanoReactor(skip_step=10,reactants=combination)
    anr.read_trajectory(f"test_{i}.traj",format=None)
    anr.analyse_trajectory()
    data[i] = {"combination":combination,"unique_smiles":anr.unique_smiles_counter[0]}
    print(data[i])


100%|██████████| 501/501 [04:49<00:00,  1.73it/s]


{'combination': ('[O]N=O', 'O=S=O'), 'unique_smiles': Counter({'[N]=O': 841, 'O=O': 448, 'O1O[N]1': 95, '[O][O]': 20, '[O]O[O]': 16, 'O=S(=O)=O': 3, '[O][N]S([O])([O])[O]': 1, 'O=S1OO1': 1, '[O][N][O]': 1})}


100%|██████████| 501/501 [05:10<00:00,  1.61it/s]


{'combination': ('[O]N=O', 'O'), 'unique_smiles': Counter({'[N]=O': 1288, 'O=O': 612, '[O][O]': 113, 'O1O[N]1': 63, '[OH]': 31, '[H]': 1})}


100%|██████████| 501/501 [03:57<00:00,  2.11it/s]

{'combination': ('O=S=O', 'O'), 'unique_smiles': Counter({'[OH]': 40, 'O=S': 1})}


In [29]:
from collections import Counter 
def combinations_for(available, possible):
    possible = [Counter(p) for p in possible]
    results = []

    def search(remaining, start, chosen):
        if not remaining:                      # everything used up
            results.append(chosen)
            return
        for i in range(start, len(possible)):
            if possible[i] <= remaining:       # enough of each element left
                search(remaining - possible[i], i, chosen + [i])

    search(Counter(available), 0, [])
    return results

available = {"O": 4, "N": 1, "S": 1}
possible = [{"N": 1, "O": 1}, {"O": 2}, {"S": 1, "O": 3}]

print(combinations_for(available, possible))

[[0, 2]]


In [46]:
# for reaction 1 : NO2 + SO2 = ? 
reaction = 0 

available = {"O": 4, "N": 1, "S": 1}

possible = [anr.get_num_atoms(smiles) for smiles in sorted(data[reaction]["unique_smiles"])]

combinations_for(available,possible)

[15:46:26] UFFTYPER: Unrecognized atom type: S_5+4 (2)


[[2, 4], [3, 4], [6]]

In [47]:
pd.Series(sorted(data[reaction]["unique_smiles"]))

0                 O1O[N]1
1                     O=O
2               O=S(=O)=O
3                 O=S1OO1
4                   [N]=O
5                 [O]O[O]
6    [O][N]S([O])([O])[O]
7               [O][N][O]
8                  [O][O]
dtype: str

In [39]:
# for reaction 2 : NO2 + H2O = ? 
reaction = 1

available = {"O": 3, "N": 1, "H": 2}

possible = [anr.get_num_atoms(smiles) for smiles in sorted(data[reaction]["unique_smiles"])]

combinations_for(available,possible)



[15:42:12] WARNING: not removing hydrogen atom without neighbors


[[0, 2, 4], [1, 2, 2, 3], [2, 2, 3, 5], [3, 4, 4]]

In [41]:
pd.Series(sorted(data[reaction]["unique_smiles"]))

0    O1O[N]1
1        O=O
2        [H]
3      [N]=O
4       [OH]
5     [O][O]
dtype: str

In [42]:
# for reaction 3 : SO2 + H2O = ? 
reaction = 2

available = {"O": 3, "S": 1, "H": 2}

possible = [anr.get_num_atoms(smiles) for smiles in sorted(data[reaction]["unique_smiles"])]

combinations_for(available,possible)


[[0, 1, 1]]

In [43]:
pd.Series(sorted(data[reaction]["unique_smiles"]))

0     O=S
1    [OH]
dtype: str

In [105]:
from reactit.reactit import ReactionGenerator

rg = ReactionGenerator(["SO2","H2O","NO2","NO","SO3","H2SO4"])
rg.iterate(4)

  0%|          | 0/9 [00:00<?, ?it/s]

['1 H2O + 1 SO3 = 1 H2SO4', '1 SO2 + 1 NO2 = 1 NO + 1 SO3']

In [88]:
data = {}
combinations = [["[N+](=O)[O-]","O"]]
for i,combination in enumerate(combinations): 
    # generate cells 
    sp = SMILESPack(combination)
    cells, charges = sp.run_multiple(1,counts=[5,5],min_dist=2) # two of each molecule, with a minimum distance of 2\AA
    atoms = cells[0]
    dyn = attach_calc_and_wall(atoms,filename=f"test_{i}.traj")

    dyn.run(5000)

    anr = AnalyseNanoReactor(skip_step=10,reactants=combination)
    anr.read_trajectory(f"test_{i}.traj",format=None)
    anr.analyse_trajectory()
    data[i] = {"combination":combination,"unique_smiles":anr.unique_smiles_counter[0]}
    print(data[i])


100%|██████████| 501/501 [05:15<00:00,  1.59it/s]

{'combination': ['[N+](=O)[O-]', 'O'], 'unique_smiles': Counter({'[N]=O': 1436, 'O=O': 522, '[O]N=O': 364, '[OH]': 310, 'O1O[N]1': 159, '[O]N(=O)=O': 106, '[O][O]': 85, '[N]O[O]': 43, 'N=O': 8, '[H]': 1, '[O]O[N][O]': 1})}


In [51]:
# reading in from xtb 

In [ ]:
anr = AnalyseNanoReactor(skip_step=10,reactants=("O=S=O","O"))
anr.read_trajectory("xtb.xyz",format="xyz")
anr.analyse_trajectory()

 27%|██▋       | 28/104 [00:09<00:25,  2.96it/s]

In [ ]:
anr.unique_smiles_counter[0]

defaultdict(<function arcskin.nanoreactor_analysis.AnalyseNanoReactor.__init__.<locals>.<lambda>()>,
            {0: Counter()})

In [100]:
# for reaction 3 : SO3 + H2O = ? 
reaction = 2

available = anr.num_reactant_atoms

possible = [anr.get_num_atoms(smiles) for smiles in sorted(anr.unique_smiles_counter[0])]

combs = combinations_for(available,possible) 

combs


[]

In [101]:
def reduce_unique_smiles(combinations,unique_smiles_counter):
    list2 = sorted(unique_smiles_counter)
    used = {i for sub in combinations for i in sub}
    list2 = [s for i, s in enumerate(list2) if i in used]
    return(list2)

reduce_unique_smiles(combs,anr.unique_smiles_counter[0])

[]

In [60]:
import os 
os.makedirs("output_molecules",exist_ok=True)
for smiles in anr.unique_smiles_counter[0]:
    smiles_to_atoms(smiles)[0].write(f"output_molecules/{smiles}.xyz")


[15:59:43] UFFTYPER: Unrecognized atom type: S_5+4 (1)
[15:59:43] UFFTYPER: Unrecognized atom type: S_5+4 (1)
[15:59:43] UFFTYPER: Unrecognized atom type: S_5+4 (1)
